# Agent v0 demo: a support agent that doesn't know the policy

## How to use this notebook
- Click a grey **code cell** and press **Shift + Enter** to run it. Run the cells top to bottom.
- Don't type messages into code cells. Cells run Python, so plain sentences cause a `SyntaxError`.
  You chat in the **text box** that appears in step 2.
- To start over: **Kernel → Restart Kernel**, then run the cells from the top again.

## How the agent works
The agent is a loop between the model (Claude) and five **tools**, which are functions that read
and change the store's database:

```
customer message ─▶ Claude decides what to do ─▶ calls a tool, e.g. get_order("ORD-T005")
                         ▲                                │
                         └──────── tool result ◀──────────┘
                    (repeats until Claude replies in plain text)
```

| Tool | What it does |
| --- | --- |
| `get_order`, `list_orders` | look up orders, items, returns and past refunds |
| `issue_refund` | creates a refund in the database |
| `update_case`, `add_case_note` | update the support case (e.g. resolved or escalated) and log notes |

The agent's whole instruction is three sentences: *"You are a customer support agent… Today is…
The customer is… The case is…"*. It has **no refund policy** and the tools don't check any, so
it does whatever seems reasonable. Where the store's rules differ from common sense (no refunds
after 30 days, over-$500 refunds need approval, a $5.99 fee on cheap items), it gets it wrong. That's
the point of v0: it's the baseline, and every failure tells us what to fix next.

## How it's evaluated
Each of the 10 **tasks** in `spec/tasks/` is a test scenario: a customer, an order, a message,
and the correct outcome. We don't grade the agent's words. We grade **what it did to the database**:
- **expected_state**: e.g. "no refund exists for ORD-T005" and "case CASE-T005 is escalated"
- **forbidden_actions**: e.g. "never called `issue_refund` on ORD-T005", even if the call failed

A task passes only if every check passes. Steps 4 and 5 below do exactly this.

In [ ]:
# Setup: make open-stack/ importable and pick the model backend.
import sys, json, sqlite3, tempfile
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

from agent.backends import make_completion
from agent.v0 import ChatSession, SYSTEM_PROMPT
from evals.harness import load_tasks, check_state, check_forbidden, run_trial
from mcp_servers.db import build_database

BACKEND = "gateway"   # Claude Code's gateway; use "litellm" with ANTHROPIC_API_KEY instead
complete, MODEL = make_completion(BACKEND)
TASKS = {t["id"]: t for t in load_tasks()}
print("Model:", MODEL)
for tid, t in TASKS.items():
    print(f"  {tid:40} {t['title']}")

## 1. Pick a scenario
Change `TASK_ID` to any id printed above, then run the cell. Each run builds a fresh database,
so nothing from a previous chat carries over.

In [ ]:
TASK_ID = "refund-over-500-requires-approval"   # try any id printed above
task = TASKS[TASK_ID]

tmp = tempfile.mkdtemp()
DB = str(build_database(Path(tmp) / "demo.sqlite", task.get("fixtures")))
CUSTOMER, CASE = task["context"]["customer_id"], task["context"]["case_id"]

def q(sql, *args):
    with sqlite3.connect(DB) as conn:
        conn.row_factory = sqlite3.Row
        return [dict(r) for r in conn.execute(sql, args)]

order = task["fixtures"]["orders"][0]
print(f"Task:     {task['title']}")
print(f"Customer: {CUSTOMER}   Case: {CASE}")
print(f"Order:    {order['id']}  {order['channel']}, paid by {order['payment_method']}, "
      f"placed {order['placed_at']}, total ${order['total']}")
print(f"Items:    {[(i['name'], i['unit_price']) for i in task['fixtures']['order_items']]}")
print(f"Returns:  {[(r['channel'], r['status']) for r in task['fixtures'].get('returns', [])]}")
print(f"Policy:   {task.get('notes', '')}")
print()
print("System prompt the agent gets:")
print(SYSTEM_PROMPT.format(today="2026-11-20", customer_id=CUSTOMER, case_id=CASE))

## 2. Chat with the agent
Run the next cell, then run the `await chat()` cell. A **text box** appears under it: type as the
customer and press **Enter**. You'll see each tool the agent calls (`->` lines), then its reply.
Type **done** to finish.

Ideas: paste the suggested opening line, then push back ("Can you add $20 for my trouble?"), or
change your story and see whether it notices.

In [ ]:
session = await ChatSession(DB, CUSTOMER, CASE, complete).__aenter__()

async def say(text):
    """Send one message as the customer and print the tool calls and the reply."""
    seen = len(session.state["tool_calls"])
    reply = await session.send(text)
    for c in session.state["tool_calls"][seen:]:
        flag = "  [error]" if c["is_error"] else ""
        print(f"  -> {c['tool']}({json.dumps(c['args'])}){flag}")
    print(f"\nAGENT: {reply}\n")

async def chat():
    """Interactive chat: type in the box, press Enter. Type 'done' to stop."""
    while True:
        text = input("YOU (type 'done' to stop): ").strip()
        if text.lower() in ("done", "quit", "exit"):
            print("Chat ended. Run the next sections to see what it changed and how it scored.")
            break
        if text:
            await say(text)

print("Session started. Suggested opening line (copy it into the chat box):\n")
print(task["conversation"][0]["user"])

In [ ]:
await chat()

In [ ]:
# Optional: send the task's exact scripted opening line without typing it (what the eval does).
# await say(task["conversation"][0]["user"])

## 3. What did it change in the database?

In [ ]:
print("Case:", q("SELECT id, status FROM cases WHERE id = ?", CASE))
print("Refunds issued by the agent:")
for r in q("SELECT id, order_id, item_id, amount, refund_method, return_channel FROM refunds WHERE created_by = 'agent'"):
    print("  ", r)
print("Case notes:")
for n in q("SELECT body FROM case_notes WHERE case_id = ?", CASE):
    print("  ", n["body"])

## 4. Grade it like the eval suite does
This checks the database against the task's `expected_state`, and the agent's tool calls against
`forbidden_actions`. **PASS** means the agent followed the store's policy. **FAIL** shows what it
did instead (`actual`). For `refund-over-500-requires-approval`, v0 usually issues the refund,
so expect FAILs here.

In [ ]:
results = check_state(DB, task["expected_state"]) + check_forbidden(
    session.state["tool_calls"], task.get("forbidden_actions", []))
for c in results:
    print("PASS" if c["passed"] else "FAIL", " ", c["check"], " actual:", c["actual"])
print()
print("TASK", "PASSED" if all(c["passed"] for c in results) else "FAILED")

In [ ]:
await session.__aexit__(None, None, None)   # close the MCP clients when you're done

## 5. Run the whole suite (about $0.50 and a few minutes)

The same as `uv run python -m evals.run --backend gateway`, one trial per task.

In [ ]:
suite = []
for tid, t in TASKS.items():
    r = await run_trial(t, 0, complete, MODEL)
    suite.append(r)
    failed = "; ".join(c["check"] for c in r["checks"] if not c["passed"])
    print("PASS" if r["passed"] else ("ERROR" if r["error"] else "FAIL"), f" {tid}", r["error"] or failed)
print(f"\n{sum(r['passed'] for r in suite)}/{len(suite)} passed, "
      f"${sum(r['usage']['cost_usd'] for r in suite):.2f}")

## 6. Review a saved trace

In [ ]:
runs = sorted((Path.cwd().parent / "runs").glob("*.jsonl"))
latest = runs[-1]
print("Trace file:", latest.name)
traces = [json.loads(l) for l in latest.read_text().splitlines()]
for r in traces:
    print("PASS" if r["passed"] else "FAIL", " ", r["task_id"])

In [ ]:
# Replay one trace: the conversation and every tool call, in order.
trace = next(r for r in traces if r["task_id"] == TASK_ID)
for m in trace["messages"]:
    if m["role"] == "user":
        print("CUSTOMER:", m["content"], "\n")
    elif m["role"] == "assistant":
        for tc in m.get("tool_calls", []):
            print("  ->", tc["function"]["name"], tc["function"]["arguments"])
        if m["content"]:
            print("AGENT:", m["content"], "\n")